In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import math
import string
import random
import time
import json
from datetime import datetime
import unicodedata #as some text is french this is important
from collections import Counter
from pickle import load, dump
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

False

In [2]:
# #We will not be using the tokens generated from Autotokenizer
# from transformers import AutoTokenizer
# fra_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
# eng_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

# print(f"Special Tokens: {eng_tokenizer.all_special_tokens}")
# print(f"Special IDs: {eng_tokenizer.all_special_ids}")
# print(f"Special Tokens: {fra_tokenizer.all_special_tokens}")
# print(f"Special IDs: {fra_tokenizer.all_special_ids}")

# tokens_file = "data/tokens.pkl" # for jupyter
# with open(tokens_file, 'rb') as file:
#     tokens = load(file)

In [3]:
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"

MIN_FREQ = 5

TABLE = str.maketrans("","",string.punctuation.replace("'","")) # as french words include '
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch_xla.device()
device

device(type='cuda')

In [4]:
seed = 42

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
# filepath = "data/eng-fra.txt"
# eng_vocab = "data/eng.json"
# fra_vocab = "data/fra.json"
filepath = "eng-fra.txt"
eng_vocab = "eng.json"
fra_vocab = "fra.json"
# vocabpath = "vocabs.json"
# !ls

In [6]:
class Vocab():
    def __init__(self, lang, min_freq):
        self.stoi = {PAD:0,SOS:1,EOS:2,UNK:3}
        self.itos = {0:PAD,1:SOS,2:EOS,3:UNK}
        self.freqs = {}
        self.nxt_idx = 4
        self.lang = lang
        self.min_freq = min_freq

    def build_vocab(self, corpus): #we should get the corpus as an iterable of all the words
        freqs = Counter(corpus)
        # VERY IMP: Counter can store in random order for diff runs so sort before vocab.
        self.freqs = dict(sorted(freqs.items(), key=lambda x:x[1], reverse=True)) #higher freq -> lower index
        for word, freq in self.freqs.items():
            if freq >= self.min_freq and word not in self.stoi:
                self.stoi[word] = self.nxt_idx
                self.itos[self.nxt_idx] = word
                self.nxt_idx += 1

    def encode(self, sent):
        tokens = [self.stoi[SOS]]
        tokens += [self.stoi.get(word, self.stoi[UNK]) for word in sent]
        tokens += [self.stoi[EOS]]

        return tokens

    def decode(self, batch):
        # batch = [bs, sl]
        out = []
        for seq in batch:
            words = []
            for token in seq:
                if token.item() == self.stoi[SOS] or token.item() == self.stoi[PAD]:
                    continue
                if token.item() == self.stoi[EOS]:
                    break
                words.append(self.itos.get(token.item(), UNK))
            out.append(" ".join(words))

        return out

    def save_vocab(self, filepath):
        with open(filepath, "w") as f:
            json.dump({"stoi": self.stoi, "itos": self.itos, "nxt_idx": self.nxt_idx}, f)

    def load_vocab(self, filepath):
        with open(filepath, "r") as f:
            data = json.load(f)
            self.stoi = data["stoi"]
            self.itos = {int(k):v for k,v in data["itos"].items()}
            self.nxt_idx = int(data["nxt_idx"])


In [7]:
class Fra2EngDataset(Dataset):
    def __init__(self, filepath, min_freq, eng_vocab_path=None, fra_vocab_path=None):
        super().__init__()
        self.fra_vocab = Vocab(lang="fra", min_freq=min_freq)
        self.eng_vocab = Vocab(lang="eng", min_freq=min_freq)
        self.pairs = []
        self.max_len = 0

        with open(filepath, 'r', encoding="utf-8") as file:
            for line in file:
                pair = line.strip().split("\t")
                if len(pair) != 2:
                    continue
                eng_tokens = self.preprocess(pair[0]).split()
                fra_tokens = self.preprocess(pair[1]).split()
                self.pairs.append((eng_tokens, fra_tokens))
                self.max_len = max([self.max_len, len(eng_tokens), len(fra_tokens)])

        if eng_vocab_path is None and fra_vocab_path is None:
            corpus = {"eng": [], "fra": []}
            for pair in self.pairs:
                corpus["eng"].extend(pair[0])
                corpus["fra"].extend(pair[1])

            self.eng_vocab.build_vocab(corpus["eng"])
            self.fra_vocab.build_vocab(corpus["fra"])

            self.eng_vocab.save_vocab(eng_vocab)
            self.fra_vocab.save_vocab(fra_vocab)

        else:
            with open(eng_vocab_path, 'r') as f:
                self.eng_vocab.load_vocab(eng_vocab_path)

            with open(fra_vocab_path, 'r') as f:
                self.fra_vocab.load_vocab(fra_vocab_path)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        eng_tokens = self.eng_vocab.encode(pair[0])
        fra_tokens = self.fra_vocab.encode(pair[1])

        eng_tensor = torch.tensor(eng_tokens, dtype=torch.long) #shifted to right by 1 due to SOS
        fra_tensor = torch.tensor(fra_tokens, dtype=torch.long) #input to encoder so no SOS

        return fra_tensor, eng_tensor # since fra -> eng

    def preprocess(self, sent):
        sent = unicodedata.normalize("NFKC", sent)
        return sent.lower().translate(TABLE)

In [8]:
# dataset = Fra2EngDataset(filepath, MIN_FREQ)
# dataset.eng_vocab.nxt_idx, dataset.fra_vocab.nxt_idx

In [9]:
# with open(vocabpath, 'wb') as f:
#     dump({"eng":dataset.eng_vocab, "fra":dataset.fra_vocab}, f)
dataset = Fra2EngDataset(filepath, MIN_FREQ, eng_vocab, fra_vocab)
dataset.eng_vocab.nxt_idx, dataset.fra_vocab.nxt_idx

(5515, 8575)

In [10]:
dataset[61110]

(tensor([  1, 431,  35,  10,  55,  46,  37, 121,   2]),
 tensor([ 1, 94, 39,  7, 51,  5, 81,  2]))

In [11]:
def custom_padding(batch, pad_idx = dataset.eng_vocab.stoi[PAD]):
    fra, eng = zip(*batch)

    max_fra = max(x.size(0) for x in fra)
    max_eng = max(x.size(0) for x in eng)

    fra_batch = torch.full((len(batch), max_fra), pad_idx, dtype=torch.long)
    eng_batch = torch.full((len(batch), max_eng), pad_idx, dtype=torch.long)

    for i in range(len(batch)):
        fra_batch[i, :fra[i].size(0)] = fra[i]
        eng_batch[i, :eng[i].size(0)] = eng[i]

    return fra_batch, eng_batch

In [12]:
train_loader = DataLoader(dataset, collate_fn=custom_padding, drop_last=True, batch_size=32, num_workers=2)

In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divible by num heads."

        self.d_model = d_model
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_o = nn.Linear(self.d_model, self.d_model, bias=False)

    def split_heads(self, x):
        # x = [bs, sl, d] -> [bs, sl, nh, dk] -> [bs, nh, sl, dk]
        batch_size, seq_len, _ = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)

    def combine_heads(self, x):
        # x = [bs, nh, sl, dk] -> [bs, sl, nh, dk] -> [bs, sl, d]
        batch_size, _, seq_len, _ = x.size()
        return x.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)

    def scaled_dot_product_attn(self, Q, K, V, mask=None):
        # Q = [bs, nh, sl, dk]
        alignment_scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # alignment_scores = [bs, nh, sl, sl] = [bs, nh, querylen, keylen]

        if mask is not None: # apply mask
            alignment_scores = alignment_scores.masked_fill(mask == 0, float('-1e9'))

        attn_weights = torch.softmax(alignment_scores, dim=-1)
        contextual_embed = attn_weights @ V
        # contextual_embed = [bs, nh, sl, dk]

        return contextual_embed, attn_weights

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        embed, attn_wts = self.scaled_dot_product_attn(Q, K, V, mask)
        output = self.W_o(self.combine_heads(embed))

        return output

In [14]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.layer1 = nn.Linear(d_model, d_ff)
        self.layer2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

In [15]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()

        # PE = sin or cos(pos / 10000^(2i/d))
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        pe = torch.zeros(max_seq_len, d_model)

        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000) / d_model)) #this is the denominator in angle
        pe[:, 0::2] = torch.sin(position * div_term) # position = [msl, 1], div = [d/2]
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)

    def forward(self, x):
        # x = embedding = [bs, sl, d]
        return x + self.pe[:x.size(1), :]

In [82]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, ):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=0.1)

    def forward(self, x, mask=None):
        # input x = embed + positional encoding = [bs, sl, d]
        x = self.norm1(x)
        z = self.mha(x, x, x, mask)
        z_norm1 = self.dropout(z) + x # residual connections

        z_norm1 = self.norm2(z_norm1)
        z = self.ffn(z_norm1)
        z_norm2 = self.dropout(z) + z_norm1

        return z_norm2

In [83]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.masked_mha = MultiHeadAttention(d_model, num_heads)
        self.cross_mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # src_mask = fra mask = padding only
        # tgt_mask = eng mask = causal + padding mask
        # non auto regressive in training

        z = self.norm1(z)
        z = self.masked_mha(x, x, x, tgt_mask)
        z1 = self.dropout(z) + x
        
        z1 = self.norm2(z1)
        z2 = self.cross_mha(z1, enc_output, enc_output, src_mask)
        z3 = self.dropout(z2) + z1

        z3 = self.norm3(z3)
        y = self.ffn(z3)
        out = self.dropout(y) + z3

        return out

In [84]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, d_ff, max_seq_len, src_vocab_size, tgt_vocab_size):
        super().__init__()

        # input
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)

        # blocks
        self.encoder_block = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.decoder_block = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])

        #output
        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(0.1)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)
        # print(type(src_mask))

        seqlen = tgt.size(1)
        no_peak_mask = (1 - torch.triu(torch.ones(1, seqlen, seqlen, device=device), diagonal=1)).bool()

        tgt_mask = tgt_mask & no_peak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):

        src_mask, tgt_mask = self.generate_mask(src, tgt)
        input_src = self.dropout(self.positional_encoding(self.src_embed(src)))
        input_tgt = self.dropout(self.positional_encoding(self.tgt_embed(tgt)))

        enc_out = input_src
        for enc_layer in self.encoder_block:
            # print(type(enc_out))
            enc_out = enc_layer(enc_out, src_mask)

        dec_out = input_tgt
        for dec_layer in self.decoder_block:
            dec_out = dec_layer(dec_out, enc_out, src_mask, tgt_mask)

        output = self.fc(dec_out)
        return output

In [19]:
D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 512
SRC_VOCAB_SIZE = dataset.fra_vocab.nxt_idx
TGT_VOCAB_SIZE = dataset.eng_vocab.nxt_idx
MAX_SEQ_LEN = dataset.max_len
EPOCHS = 75

In [20]:
model = Transformer(D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, MAX_SEQ_LEN, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE)

In [21]:
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [22]:
def train_one_epoch(model, criterion, optimizer, train_loader):
    progress = tqdm(train_loader, desc="Data", total=len(train_loader))
    train_losses = []

    model.train()

    for src, tgt in progress:
        src, tgt = src.to(device), tgt.to(device)
        # src = [bs, sl], tgt = [bs, sl]

        tgt_input_decoder = tgt[:,:-1] # skip EOS as we dont want to predict anything by sending this as input
        tgt_input_loss = tgt[:,1:] # compare w1 with w1 not SOS

        output = model(src, tgt_input_decoder)

        predicted = output.contiguous().view(-1, TGT_VOCAB_SIZE) # output = [bs, sl, tgt_vocab_size]
        target = tgt_input_loss.contiguous().view(-1)

        loss = criterion(predicted, target)
        train_losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    return sum(train_losses) / len(train_losses)

In [23]:
from google.colab import drive
drive.mount('/content/drive/')

save_dir = '/content/drive/MyDrive/transformer'
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive/


In [29]:
def save_checkpoint(model, epoch, train_losses):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'loss': train_losses
    }
    filepath = os.path.join(save_dir, f"checkpt{epoch}.pth")
    torch.save(checkpoint, filepath)

    old_checkpoint = os.path.join(save_dir, f"checkpt{epoch-1}.pth")
    if os.path.exists(old_checkpoint):
        os.remove(old_checkpoint)
    print(f"Checkpoint saved to {filepath} at epoch {epoch}\n")

In [25]:
def load_checkpoint(model, checkpt_path):
    checkpt = torch.load(checkpt_path, map_location=device)
    model.load_state_dict(checkpt['model_state_dict'])
    epoch = checkpt['epoch']
    losses = checkpt['loss']

    return model, optimizer, epoch, losses

In [26]:
# checkpt_path = "checkpoints/checkpt3.pth"
checkpt_path = os.path.join(save_dir, "checkpt3.pth")
model, i, training_losses = load_checkpoint(model, checkpt_path)

In [31]:
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# training_losses = []

for epoch in range(i+1, EPOCHS+1):
    strt = time.time()
    print(f"Start time: {datetime.now()}")
    loss = train_one_epoch(model, criterion, optimizer, train_loader)
    end = time.time()
    training_losses.append(loss)
    print(f"Epoch {epoch}: Loss = {loss} | Time Taken = {end-strt} seconds")
    save_checkpoint(model, epoch, training_losses)

Start time: 2026-03-26 05:32:46.265605


Data: 100%|██████████| 4245/4245 [01:25<00:00, 49.87it/s]


Epoch 4: Loss = 3.8508630373452664 | Time Taken = 85.13076686859131 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt4.pth at epoch 4

Start time: 2026-03-26 05:34:11.461717


Data: 100%|██████████| 4245/4245 [01:24<00:00, 50.16it/s]


Epoch 5: Loss = 3.7384533236248614 | Time Taken = 84.63422083854675 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt5.pth at epoch 5

Start time: 2026-03-26 05:35:36.166812


...

Epoch 73: Loss = 3.0502392431030003 | Time Taken = 83.43127250671387 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt73.pth at epoch 73

Start time: 2026-03-26 07:11:33.204764


Data: 100%|██████████| 4245/4245 [01:23<00:00, 50.67it/s]


Epoch 74: Loss = 3.0594779433294237 | Time Taken = 83.77551102638245 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt74.pth at epoch 74

Start time: 2026-03-26 07:12:57.033969


Data: 100%|██████████| 4245/4245 [01:23<00:00, 50.69it/s]


Epoch 75: Loss = 3.051141443185166 | Time Taken = 83.7476692199707 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt75.pth at epoch 75



In [ ]:
def decode_tgt(tgt_tensor, tgt_vocab):
  # tgt = [bs, sl, vocab] -> not in probs
  tokens = tgt_tensor.argmax(-1)
  return tgt_vocab.decode(tokens.squeeze(-1))

def decode_src(src_tensor, src_vocab):
  # src = [bs, sl]
  return src_vocab.decode(src_tensor)

In [ ]:
batch = []
for i in range(32):
  s, t = dataset[i]
  batch.append((s,t))
src1, tgt1 = custom_padding(batch)

In [ ]:
# src1, tgt1 = next(iter(train_loader))
out = model(src1, tgt1)
# torch.topk(out, 2, dim=-1).values.shape
decode_tgt(out, dataset.eng_vocab)
# torch.softmax(out, dim=-1)
# decode_src(src1, dataset.fra_vocab)
# decode_src(tgt1, dataset.eng_vocab)